### ✅ BLOQUE 0 — Setup y verificación de archivos (Markdown)

Qué hace y por qué:
Antes de estimar un VAR necesitamos asegurarnos de que el notebook está leyendo desde la carpeta correcta y que todos los archivos esperados están disponibles. Este bloque define rutas (raw/) de forma robusta, lista los archivos y te confirma que estás apuntando al lugar correcto. Esto evita errores típicos de “file not found” o de rutas relativas cuando corres el notebook desde VS Code.

In [178]:
# === BLOQUE 0: Setup y verificación de archivos ===
from pathlib import Path
import pandas as pd
import numpy as np

# Ruta base: carpeta donde está este notebook
BASE_DIR = Path.cwd()

# Si por alguna razón corres desde otra carpeta, puedes forzar:
# BASE_DIR = Path("/ruta/a/tu/proyecto/Banxico/recuadro")

RAW_DIR = BASE_DIR / "raw"

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("\nArchivos en raw/:")
for p in sorted(RAW_DIR.glob("*")):
    print(" -", p.name)

# Lista mínima esperada (ajústala si cambias nombres)
expected = [
    "DEXMXUS.csv",
    "DGS10.csv",
    "VIXCLS.csv",
    "EMBIG.csv",
    "INPC.csv",
    "qqq_us_d.csv",
    "spy_us_d.csv",
    # opcionales:
    "NASDAQ100.csv",
    "SP500.csv",
    # "IGAE_2.xlsx"  # lo dejamos para después
]

missing = [f for f in expected if not (RAW_DIR / f).exists()]
if missing:
    print("\n⚠️ Faltan archivos esperados:")
    for m in missing:
        print(" -", m)
else:
    print("\n✅ Están todos los archivos esperados (según tu lista).")

BASE_DIR: /Users/emiliahernandez/Desktop/Banxico/recuadro
RAW_DIR: /Users/emiliahernandez/Desktop/Banxico/recuadro/raw

Archivos en raw/:
 - DEXMXUS.csv
 - DGS10.csv
 - EMBIG.csv
 - IGAE_2.xlsx
 - INPC.csv
 - NASDAQ100.csv
 - SP500.csv
 - VIXCLS.csv
 - cetes_28_banxico_diario.csv
 - qqq_us_d.csv
 - spy_us_d.csv

✅ Están todos los archivos esperados (según tu lista).


### ✅ BLOQUE 1 — Funciones de lectura + carga de CSVs (Markdown)

Qué hace y por qué:
Tus archivos vienen de distintas fuentes (FRED, Stooq, BCRP). Aunque todos son “CSV”, no tienen el mismo formato:
- FRED suele venir como DATE, VALUE (mensual o diario).
- Stooq viene como Date, Open, High, Low, Close, Volume (diario).
- BCRP puede traer Fecha, Valor y formatos de fecha distintos.

Este bloque define funciones de lectura “tolerantes” para que todos queden con dos columnas estándar:
- date (datetime)
- value (float)
y además deja cada serie en un diccionario series.

In [179]:
from pathlib import Path
import pandas as pd
import numpy as np

def read_any_csv_to_date_value(path: Path) -> pd.DataFrame:
    """
    Lector robusto para CSVs diversos (FRED, BCRP, exportados de Excel, etc.)
    Devuelve columnas: date, value
    """
    # 1) Probar encodings típicos
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]
    last_err = None
    df = None

    for enc in encodings:
        try:
            # 2) detectar separador automáticamente + tolerar líneas malas
            df = pd.read_csv(
                path,
                encoding=enc,
                sep=None,              # autodetecta sep (coma/; /tab)
                engine="python",
                on_bad_lines="skip",   # ignora líneas problemáticas
            )
            # si leyó algo razonable, salimos
            if df is not None and df.shape[1] >= 2 and len(df) > 0:
                break
        except Exception as e:
            last_err = e
            df = None

    if df is None:
        raise ValueError(f"No pude leer {path.name}. Último error: {last_err}")

    # Normaliza nombres
    df.columns = [str(c).strip() for c in df.columns]

    # 3) detectar columna fecha
    date_candidates = [c for c in df.columns if c.lower() in ["date", "fecha", "observation_date", "time", "period", "periodo", "día", "dia", "d\u00eda"]]
    if date_candidates:
        date_col = date_candidates[0]
    else:
        # fallback: primera columna
        date_col = df.columns[0]

    # 4) convertir a datetime (probar dayfirst y no-dayfirst)
    tmp = df[[date_col]].copy()
    d1 = pd.to_datetime(tmp[date_col], errors="coerce", dayfirst=True)
    d2 = pd.to_datetime(tmp[date_col], errors="coerce", dayfirst=False)
    # elige la que tenga más fechas válidas
    date_parsed = d1 if d1.notna().sum() >= d2.notna().sum() else d2

    # 5) elegir una columna numérica como "value"
    other_cols = [c for c in df.columns if c != date_col]
    best_col = None
    best_non_na = -1

    for c in other_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        non_na = s.notna().sum()
        if non_na > best_non_na:
            best_non_na = non_na
            best_col = c

    if best_col is None or best_non_na <= 0:
        raise ValueError(f"No encontré columna numérica usable en {path.name}. Columnas: {df.columns.tolist()}")

    out = pd.DataFrame({"date": date_parsed, "value": pd.to_numeric(df[best_col], errors="coerce")})
    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    return out
def read_fred_monthly_csv(path: Path, value_col: str) -> pd.DataFrame:
    """
    Lee CSV de FRED que ya viene mensual (observation_date, <SERIE>).
    Devuelve columnas: date, <SERIE> con date en month-start (YYYY-MM-01).
    """
    df = pd.read_csv(path)
    # FRED típico: observation_date, DGS10 (o VIXCLS)
    df = df.rename(columns={"observation_date": "date"}).copy()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    # Estandariza a month-start para que siempre haga match en merges
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp("MS")

    out = df[["date", value_col]].dropna().sort_values("date").reset_index(drop=True)
    return out

# === prueba rápida: intenta leer todos los CSV en raw/ ===
series = {}
csv_files = sorted((RAW_DIR).glob("*.csv"))

print("CSV encontrados:", [p.name for p in csv_files])

for p in csv_files:
    try:
        series[p.stem] = read_any_csv_to_date_value(p)
        df = series[p.stem]
        print(f"✅ {p.name:15s} -> {df.shape} | {df['date'].min().date()} .. {df['date'].max().date()}")
    except Exception as e:
        print(f"❌ {p.name:15s} -> {e}")

CSV encontrados: ['DEXMXUS.csv', 'DGS10.csv', 'EMBIG.csv', 'INPC.csv', 'NASDAQ100.csv', 'SP500.csv', 'VIXCLS.csv', 'cetes_28_banxico_diario.csv', 'qqq_us_d.csv', 'spy_us_d.csv']
✅ DEXMXUS.csv     -> (387, 2) | 1993-01-12 .. 2026-01-02
✅ DGS10.csv       -> (337, 2) | 1998-01-01 .. 2026-01-01
❌ EMBIG.csv       -> No pude leer EMBIG.csv. Último error: Could not determine delimiter
❌ INPC.csv        -> No encontré columna numérica usable en INPC.csv. Columnas: ['Instituto Nacional de Estadística y Geografía', 'Unnamed: 1']
✅ NASDAQ100.csv   -> (481, 2) | 1986-01-01 .. 2026-01-01
✅ SP500.csv       -> (2610, 2) | 2016-02-22 .. 2026-02-20
✅ VIXCLS.csv      -> (433, 2) | 1990-01-01 .. 2026-01-01
✅ cetes_28_banxico_diario.csv -> (1014, 2) | 2006-09-19 .. 2026-02-17
✅ qqq_us_d.csv    -> (6779, 2) | 1999-03-10 .. 2026-02-20
✅ spy_us_d.csv    -> (5279, 2) | 2005-02-25 .. 2026-02-20


/var/folders/5x/v_n3jdfd003f7mqj3m9j1k0h0000gn/T/ipykernel_56496/2782867652.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d1 = pd.to_datetime(tmp[date_col], errors="coerce", dayfirst=True)
/var/folders/5x/v_n3jdfd003f7mqj3m9j1k0h0000gn/T/ipykernel_56496/2782867652.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d2 = pd.to_datetime(tmp[date_col], errors="coerce", dayfirst=False)
/var/folders/5x/v_n3jdfd003f7mqj3m9j1k0h0000gn/T/ipykernel_56496/2782867652.py:48: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d1 = pd.to_datetime(tmp[date_col], errors="coerce", dayfirst=True)
/var/folders/5x/v_n3jdfd003f

In [180]:
df

,date,value
0,2005-02-25,92.7949
1,2005-02-28,93.4735
2,2005-03-01,93.1845
3,2005-03-02,93.1749
4,2005-03-03,93.8716
...,...,...
5274,2026-02-13,681.6900
5275,2026-02-17,680.1400
5276,2026-02-18,684.0200
5277,2026-02-19,683.8400


Diagnóstico de archivos que faltan

In [181]:
# === BLOQUE 1.2A: Diagnóstico rápido de INPC y EMBIG ===
from pathlib import Path

INPC_PATH = RAW_DIR / "INPC.csv"
EMBIG_PATH = RAW_DIR / "EMBIG.csv"

def head_text(path: Path, n=25):
    print(f"\n--- {path.name} (primeras {n} líneas) ---")
    with open(path, "r", errors="replace") as f:
        for i in range(n):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:02d}: {line.rstrip()}")

head_text(INPC_PATH, n=25)
head_text(EMBIG_PATH, n=25)


--- INPC.csv (primeras 25 líneas) ---
01: "Instituto Nacional de Estad�stica y Geograf�a",
02: "�ndice Nacional de Precios al Consumidor y sus Componentes",
03: 
04: "T�tulo"," �ndice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualizaci�n de Canasta y Ponderadores 2024 (mensual), Resumen, Principales �ndices, Precios al Consumidor (INPC)"," �ndice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualizaci�n de Canasta y Ponderadores 2024 (mensual), Resumen, Principales �ndices, Precios al Consumidor (INPC), Subyacente"," �ndice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualizaci�n de Canasta y Ponderadores 2024 (mensual), Resumen, Sub�ndices subyacente y complementarios, Precios al Consumidor (INPC), Subyacente, Mercanc�as"," �ndice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualizaci�n de Canasta y Ponderadores 2024 (mensual), Resumen, Sub�ndices subyacente y complementarios, Pr

In [182]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

INPC_PATH = RAW_DIR / "INPC.csv"
EMBIG_PATH = RAW_DIR / "EMBIG.csv"

# ---------- EMBIG: directo ----------

def read_embig_from_bcrp_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        encoding="latin-1",
        skiprows=4,
        header=None,
        names=["date_raw", "value_raw"],
        engine="python"
    ).dropna(subset=["date_raw", "value_raw"]).copy()

    # Traducción meses ES -> EN para que %b funcione
    month_map = {
        "Ene": "Jan", "Feb": "Feb", "Mar": "Mar", "Abr": "Apr",
        "May": "May", "Jun": "Jun", "Jul": "Jul", "Ago": "Aug",
        "Sep": "Sep", "Oct": "Oct", "Nov": "Nov", "Dic": "Dec",
    }

    s = df["date_raw"].astype(str).str.strip()
    # reemplaza el bloque de mes (3 letras) por equivalente en inglés
    s = s.str.replace(
        r"(\d{2})([A-Za-zÁÉÍÓÚáéíóúÑñ]{3})(\d{2})",
        lambda m: m.group(1) + month_map.get(m.group(2).title(), m.group(2).title()) + m.group(3),
        regex=True
    )

    df["date"] = pd.to_datetime(s, format="%d%b%y", errors="coerce")
    df["value"] = pd.to_numeric(df["value_raw"], errors="coerce")

    out = df[["date", "value"]].dropna(subset=["date", "value"]).sort_values("date").reset_index(drop=True)
    return out

# ---------- INPC: detectar dónde empiezan los datos ----------
MONTHS_ES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12
}

def parse_inpc_period(s: str):
    """
    Convierte strings tipo 'Ene 1970' o 'Ene-1970' a Timestamp del 1er día del mes.
    """
    s = str(s).strip().lower().replace(".", "")
    # patrones comunes: "Ene 1970", "Ene-1970", "Ene1970"
    m = re.search(r"([a-zñ]{3})[\s\-]?(\d{4})", s)
    if not m:
        return pd.NaT
    mon = MONTHS_ES.get(m.group(1)[:3], None)
    yr = int(m.group(2))
    if mon is None:
        return pd.NaT
    return pd.Timestamp(year=yr, month=mon, day=1)

def find_inpc_data_start_line(path: Path, encoding="latin-1", max_lines=5000) -> int:
    """
    Encuentra la primera línea que 'parece' dato:
    algo que contenga un mes (Ene/Feb/...) y un año 4 dígitos,
    y además una coma después (porque el archivo es CSV).
    """
    pat = re.compile(r'(Ene|Feb|Mar|Abr|May|Jun|Jul|Ago|Sep|Oct|Nov|Dic)[\s\-]?\d{4}', re.IGNORECASE)
    with open(path, "r", encoding=encoding, errors="replace") as f:
        for i, line in enumerate(f):
            if i > max_lines:
                break
            if pat.search(line) and "," in line:
                return i
    return 0

def read_inpc_inegi_csv(path: Path) -> pd.DataFrame:
    # 1) encontrar inicio real de datos
    start = find_inpc_data_start_line(path, encoding="latin-1")
    # 2) leer desde ahí: esperamos 2 columnas (Periodo, Valor) al menos para el INPC general
    df = pd.read_csv(
        path,
        encoding="latin-1",
        skiprows=start,
        header=None,
        engine="python",
        on_bad_lines="skip"
    )
    # Nos quedamos con las primeras 2 columnas
    df = df.iloc[:, :2]
    df.columns = ["period_raw", "value_raw"]
    # 3) parseo
    df["date"] = df["period_raw"].apply(parse_inpc_period)
    df["value"] = pd.to_numeric(df["value_raw"], errors="coerce")
    out = df[["date", "value"]].dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # Filtro: en INPC es normal que empiece 1970, pero nosotros luego recortamos muestra
    return out


# ---- ejecutar y meter a tu dict ----
series["EMBIG"] = read_embig_from_bcrp_csv(EMBIG_PATH)
print("✅ EMBIG:", series["EMBIG"].shape, "|", series["EMBIG"]["date"].min().date(), "..", series["EMBIG"]["date"].max().date())

series["INPC"] = read_inpc_inegi_csv(INPC_PATH)
print("✅ INPC:", series["INPC"].shape, "|", series["INPC"]["date"].min().date(), "..", series["INPC"]["date"].max().date())

# vistazo
print("\nEMBIG head/tail")
display(series["EMBIG"].head(3))
display(series["EMBIG"].tail(3))

print("\nINPC head/tail")
display(series["INPC"].head(3))
display(series["INPC"].tail(3))

✅ EMBIG: (6740, 2) | 1998-01-02 .. 2026-02-19
✅ INPC: (337, 2) | 1998-01-01 .. 2026-01-01

EMBIG head/tail


,date,value
0,1998-01-02,403
1,1998-01-05,422
2,1998-01-06,432


,date,value
6737,2026-02-17,217
6738,2026-02-18,214
6739,2026-02-19,215



INPC head/tail


,date,value
0,1998-01-01,34.003924
1,1998-02-01,34.599238
2,1998-03-01,35.004533


,date,value
334,2025-11-01,142.645
335,2025-12-01,143.042
336,2026-01-01,143.588


### Lectura del IGAE desde archivo Excel

El archivo del IGAE tiene una estructura típica de INEGI:

- Varias filas de metadatos arriba
- Años en columnas
- Meses en columnas (Enero, Febrero, …)
- Los valores están en formato "wide"

Para usarlo en el VAR necesitamos:

1. Extraer únicamente el renglón del **IGAE total**
2. Convertir la tabla de formato ancho (meses como columnas) a formato largo
3. Construir una variable de fecha mensual
4. Ordenar y dejar el índice como `date`
5. Conservar el nivel del índice (base 2018 = 100)

Esto nos permitirá después calcular crecimiento mensual o anual.

In [183]:
import pandas as pd
import numpy as np
import re

# ---- helpers: mes robusto (acepta "Eneroᵖ", "Enero*", "Enero P", etc.) ----
MONTH_PREFIX = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct":10, "nov":11, "dic":12
}

def month_num_from_header(x: str):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    # deja solo letras (incluye acentos/ñ)
    s = re.sub(r"[^a-záéíóúüñ]", "", s)
    # normaliza septiembre (a veces "setiembre")
    if s.startswith("set"):
        s = "sep" + s[3:]
    pref = s[:3]
    return MONTH_PREFIX.get(pref)

def read_igae_excel(path, sheet_name="Tabulado", concept_name="Total"):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # fila donde empieza la tabla (Concepto)
    concept_row = raw.index[raw.iloc[:,0].astype(str).str.contains("Concepto", na=False)].tolist()
    if not concept_row:
        raise ValueError("No encontré la fila 'Concepto' en la primera columna.")
    start = concept_row[0]

    tab = raw.iloc[start:].copy().reset_index(drop=True)

    # localizar fila de años (busca 1993..2026 en primeras filas)
    year_row_idx = None
    for r in range(0, 10):
        row = tab.iloc[r].astype(str).str.strip()
        if row.str.contains(r"(19|20)\d{2}", na=False).any():
            year_row_idx = r
            break
    if year_row_idx is None:
        raise ValueError("No pude ubicar una fila con años (1993...2026).")

    year_row = tab.iloc[year_row_idx].astype(str).str.strip()

    # fila de meses: normalmente está abajo del año, pero puede variar;
    # probamos year_row_idx+1, si no sirve, usamos year_row_idx+2.
    month_row_idx = min(year_row_idx + 1, tab.shape[0] - 1)
    header_months = tab.iloc[month_row_idx].astype(str).str.strip()

    # datos empiezan más abajo
    data_start = max(month_row_idx + 1, year_row_idx + 2)
    data = tab.iloc[data_start:].copy()
    data.columns = range(data.shape[1])

    # encontrar el renglón del concepto (Total)
    concept_col = 0
    cm = data[concept_col].astype(str).str.strip().eq(concept_name)
    if not cm.any():
        cm = data[concept_col].astype(str).str.contains(r"^Total$", na=False)
    if not cm.any():
        raise ValueError(f"No encontré el concepto '{concept_name}' en la primera columna.")
    row_idx = cm.idxmax()
    row_vals = data.loc[row_idx]

    # detectar columnas-año (extrae 1993..2026 aunque venga como "2025P")
    years_by_col = {}
    for c, txt in enumerate(year_row):
        m = re.search(r"(19|20)\d{2}", str(txt))
        if m:
            years_by_col[c] = int(m.group(0))
    if not years_by_col:
        raise ValueError("No detecté columnas-año en los encabezados.")

    out = []
    ncols = data.shape[1]

    for c0, y in sorted(years_by_col.items(), key=lambda x: x[0]):
        # buscar el primer mes a la derecha de la columna del año (rango amplio)
        start_month_col = None
        for c in range(c0, min(c0 + 30, ncols)):
            if month_num_from_header(header_months.iloc[c]) == 1:  # Enero (aunque sea "Eneroᵖ")
                start_month_col = c
                break
        if start_month_col is None:
            continue  # no pude ubicar Enero para ese año

        # leer 12 meses consecutivos
        for c in range(start_month_col, min(start_month_col + 12, ncols)):
            mnum = month_num_from_header(header_months.iloc[c])
            if mnum is None:
                break
            v = row_vals.iloc[c]
            if pd.isna(v):
                continue
            out.append({"date": pd.Timestamp(y, mnum, 1), "IGAE": float(v)})

    igae = (pd.DataFrame(out)
            .sort_values("date")
            .drop_duplicates("date")
            .reset_index(drop=True))

    if igae.empty:
        raise ValueError("IGAE salió vacío: no encontré meses/valores en los bloques-año.")

    return igae

In [184]:
igae = read_igae_excel("raw/IGAE_2.xlsx")

igae.head(), igae.tail(), igae.shape

/var/folders/5x/v_n3jdfd003f7mqj3m9j1k0h0000gn/T/ipykernel_56496/979165225.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if row.str.contains(r"(19|20)\d{2}", na=False).any():


(        date       IGAE
 0 1993-01-01  55.434736
 1 1993-02-01  56.456971
 2 1993-03-01  58.900549
 3 1993-04-01  57.135844
 4 1993-05-01  57.891853,
           date        IGAE
 390 2025-07-01  105.741081
 391 2025-08-01  104.892898
 392 2025-09-01  103.181845
 393 2025-10-01  107.900786
 394 2025-11-01  107.822450,
 (395, 2))

## ✅ Markdown 1 — Datasets ya cargados (nombres actuales en el notebook)

En este notebook, las series ya quedaron cargadas en un diccionario llamado series, donde cada DataFrame se llama igual que el archivo (sin .csv) y tiene estructura estándar date, value. En particular, tienes: series["DEXMXUS"], series["DGS10"], series["NASDAQ100"], series["SP500"], series["VIXCLS"], series["qqq_us_d"], series["spy_us_d"]. Además, el IGAE lo tienes como un DataFrame separado llamado igae (con columnas date, value) y el INPC ya lo estás leyendo también como inpc (en tu screenshot se ve el head/tail ya bien).

In [185]:
series

{'DEXMXUS':           date    value
 0   1993-01-12   3.1083
 1   1994-01-01   3.1078
 2   1994-01-02   3.1218
 3   1994-01-03   3.3026
 4   1994-01-04   3.3495
 ..         ...      ...
 382 2025-01-10  18.4246
 383 2025-01-11  18.4193
 384 2025-01-12  18.0708
 385 2026-01-01  17.6446
 386 2026-01-02      NaN
 
 [387 rows x 2 columns],
 'DGS10':           date  value
 0   1998-01-01   5.54
 1   1998-01-02   5.57
 2   1998-01-03   5.65
 3   1998-01-04   5.64
 4   1998-01-05   5.65
 ..         ...    ...
 332 2025-01-09   4.12
 333 2025-01-10   4.06
 334 2025-01-11   4.09
 335 2025-01-12   4.14
 336 2026-01-01   4.21
 
 [337 rows x 2 columns],
 'NASDAQ100':           date     value
 0   1986-01-01    132.93
 1   1986-01-02    140.43
 2   1986-01-03    148.86
 3   1986-01-04    154.91
 4   1986-01-05    163.16
 ..         ...       ...
 476 2025-01-09  24679.99
 477 2025-01-10  25858.13
 478 2025-01-11  25434.89
 479 2025-01-12  25249.85
 480 2026-01-01  25552.39
 
 [481 rows x 2 columns]

In [186]:
igae

,date,IGAE
0,1993-01-01,55.434736
1,1993-02-01,56.456971
2,1993-03-01,58.900549
3,1993-04-01,57.135844
4,1993-05-01,57.891853
...,...,...
390,2025-07-01,105.741081
391,2025-08-01,104.892898
392,2025-09-01,103.181845
393,2025-10-01,107.900786


### Inciso (2): construir retornos log y tech_rel = r_QQQ - r_SPY (paso por paso, con el porqué)

Primero vamos a definir qué series se usan para retornos: los retornos log se calculan para variables de precios/índices financieros (como qqq_us_d y spy_us_d) porque son series positivas y el retorno log (ln(P_t) - ln(P_{t-1})) se interpreta como un cambio porcentual aproximado y es estándar en finanzas. En cambio, variables macro como DGS10 (tasa), EMBIG (spread), DEXMXUS (tipo de cambio) o VIXCLS (índice) podemos tratarlas después según el diseño del VAR (niveles vs cambios), pero en este inciso nos enfocamos en retornos log de QQQ y SPY.

Luego hacemos el cálculo en 4 subpasos: (1) alinear frecuencia y fechas: como QQQ y SPY son diarios y tienen calendarios distintos por días inhábiles, los juntamos por fecha (inner join) para quedarnos con las fechas donde existen ambos y evitar retornos “fantasma” por días faltantes; (2) elegir el precio: si tu dataset ya trae una sola columna value, asumimos que es el “close” (o equivalente) y lo usamos como 
𝑃
𝑡
P
t
	​

; (3) computar retornos log: creamos r_QQQ = log(QQQ_t) - log(QQQ_{t-1}) y lo mismo para SPY, cuidando que si hay ceros o valores faltantes se eliminen antes; (4) construir el diferencial tecnológico: definimos tech_rel = r_QQQ - r_SPY, que captura el rendimiento relativo de “tech-heavy” (QQQ) frente al mercado amplio (SPY). Esto es útil porque en lugar de meter dos retornos altamente correlacionados al VAR, metes una variable que ya está “net of market” y suele ser más interpretable como shock sectorial/rotación hacia tecnología.

In [187]:
# ============================================================
# (2) Retornos log y tech_rel = r_QQQ - r_SPY
# ============================================================

import numpy as np
import pandas as pd

def make_log_returns(df, name="asset"):
    """
    df: DataFrame con columnas ['date','value'] (precio/índice)
    Regresa df con columnas: date, value, r_{name}
    """
    out = df.copy().sort_values("date")
    out = out.dropna(subset=["date", "value"])
    out = out[out["value"] > 0]  # log requiere positivos
    
    out[f"r_{name}"] = np.log(out["value"]).diff()
    return out.dropna(subset=[f"r_{name}"])[["date", "value", f"r_{name}"]]

# 1) Tomamos QQQ y SPY desde el diccionario series (ya estandarizados a date/value)
qqq = series["qqq_us_d"].copy()
spy = series["spy_us_d"].copy()

# 2) Retornos log individuales
qqq_r = make_log_returns(qqq, name="QQQ")
spy_r = make_log_returns(spy, name="SPY")

# 3) Alinear por fecha (solo fechas donde hay ambos retornos)
rets = pd.merge(
    qqq_r[["date", "r_QQQ"]],
    spy_r[["date", "r_SPY"]],
    on="date",
    how="inner"
).sort_values("date").reset_index(drop=True)

# 4) Spread relativo tech vs market
rets["tech_rel"] = rets["r_QQQ"] - rets["r_SPY"]

# 5) Quick sanity checks
print("Rangos y dimensiones:")
print("QQQ returns:", qqq_r["date"].min().date(), "..", qqq_r["date"].max().date(), "| n=", len(qqq_r))
print("SPY returns:", spy_r["date"].min().date(), "..", spy_r["date"].max().date(), "| n=", len(spy_r))
print("MERGE rets  :", rets["date"].min().date(), "..", rets["date"].max().date(), "| n=", len(rets))

rets.head(), rets.tail()

Rangos y dimensiones:
QQQ returns: 1999-03-11 .. 2026-02-20 | n= 6778
SPY returns: 2005-02-28 .. 2026-02-20 | n= 5278
MERGE rets  : 2005-02-28 .. 2026-02-20 | n= 5278


(        date     r_QQQ     r_SPY  tech_rel
 0 2005-02-28  0.002257  0.007286 -0.005030
 1 2005-03-01 -0.002475 -0.003097  0.000621
 2 2005-03-02  0.001349 -0.000103  0.001452
 3 2005-03-03  0.007856  0.007450  0.000407
 4 2005-03-04 -0.002980  0.002974 -0.005954,
            date     r_QQQ     r_SPY  tech_rel
 5273 2026-02-13 -0.023505 -0.018243 -0.005262
 5274 2026-02-17 -0.003428 -0.002276 -0.001152
 5275 2026-02-18  0.006223  0.005688  0.000534
 5276 2026-02-19  0.001162 -0.000263  0.001425
 5277 2026-02-20 -0.004472 -0.002225 -0.002247)

### CETES 28d (tasa México): limpieza y agregación a mensual

Este bloque:
1) Lee el archivo `cetes_28_banxico_diario.csv` con columnas `fecha` y `cetes_28`.
2) Convierte fecha a `datetime` y tasa a numérico.
3) Pasa la serie a frecuencia mensual usando el **último dato disponible del mes** (end-of-month), porque es una tasa observada en mercado y queremos un “estado” del mes para el VAR.
4) Devuelve un DataFrame con columnas: `date` (inicio de mes) y `CETES28` (tasa %).

In [188]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("raw")  # o el que ya estés usando
# Si en tu notebook ya tienes RAW_DIR definido, borra esta línea.

def read_cetes28_daily(path: str | Path) -> pd.DataFrame:
    """
    Lee CETES 28d diario.
    Espera columnas: fecha, cetes_28  (según tu archivo)
    Devuelve: ['date','CETES28'] diario (date en datetime, CETES28 float)
    """
    path = Path(path)
    df = pd.read_csv(path)

    # Normaliza nombres por si vienen raros
    df.columns = [c.strip().lower() for c in df.columns]

    # Ajusta nombres esperados
    # (en tu archivo se ve: fecha,cetes_28)
    if "fecha" not in df.columns or "cetes_28" not in df.columns:
        raise ValueError(f"Columnas inesperadas en {path.name}: {df.columns.tolist()}")

    out = df.rename(columns={"fecha": "date", "cetes_28": "CETES28"}).copy()

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["CETES28"] = pd.to_numeric(out["CETES28"], errors="coerce")

    out = out.dropna(subset=["date", "CETES28"]).sort_values("date").reset_index(drop=True)
    return out

def daily_to_monthly_last(df_daily: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Convierte diario a mensual (último dato del mes).
    Regresa date como inicio de mes (MS).
    """
    d = df_daily.copy()
    d["month"] = d["date"].dt.to_period("M").dt.to_timestamp()  # inicio de mes
    # Tomamos el último dato observado dentro de cada mes
    m = d.sort_values("date").groupby("month", as_index=False)[col].last()
    return m.rename(columns={"month": "date"})

# --- Ejecuta ---
cetes_d = read_cetes28_daily(RAW_DIR / "cetes_28_banxico_diario.csv")
cetes_m = daily_to_monthly_last(cetes_d, "CETES28")

print("CETES diario:", cetes_d.shape, "|", cetes_d["date"].min().date(), "->", cetes_d["date"].max().date())
print("CETES mensual:", cetes_m.shape, "|", cetes_m["date"].min().date(), "->", cetes_m["date"].max().date())

cetes_m.head(), cetes_m.tail()

CETES diario: (1014, 2) | 2006-09-19 -> 2026-02-17
CETES mensual: (234, 2) | 2006-09-01 -> 2026-02-01


(        date  CETES28
 0 2006-09-01     7.05
 1 2006-10-01     7.04
 2 2006-11-01     7.05
 3 2006-12-01     7.02
 4 2007-01-01     7.05,
           date  CETES28
 229 2025-10-01     7.10
 230 2025-11-01     7.15
 231 2025-12-01     7.07
 232 2026-01-01     6.95
 233 2026-02-01     6.84)

## Markdown — Bloque 3: “Resample a frecuencia del VAR y construir el panel”

En este bloque convertimos todas las series a una misma frecuencia (mensual) para poder estimar un VAR con un panel consistente. La lógica es: las series financieras diarias (rets con r_QQQ, r_SPY, tech_rel) se agregan a mensual usando la propiedad de los retornos log: el retorno del mes es la suma de retornos diarios del mes. Esto mantiene la interpretación económica correcta (retorno mensual). En cambio, series macro ya mensuales como igae e inpc se dejan en mensual y solo se estandariza el sello de fecha a “inicio de mes” (MS). Finalmente, juntamos todo en un solo DataFrame (df_var) alineado por fecha, y dejamos listo el dataset que alimentará el VAR (todavía sin estimar).

## ✅ Bloque 3 — Panel mensual (alinear series) + muestra de estimación

Objetivo:
1) Llevar todas las variables a frecuencia mensual con una regla clara:
   - Retornos log diarios → retornos log mensuales = suma dentro del mes.
   - Series de nivel (tipo de cambio, tasa 10y, VIX, EMBI) → nivel mensual = último dato disponible del mes.
   - Series ya mensuales (INPC, IGAE) → se dejan como están (date en inicio de mes).

2) Unir todo en un solo DataFrame mensual `df_m`, alineado por `date`.

3) Definir la muestra de estimación desde 2005-03 (porque SPY empieza en 2005-02 y el primer retorno mensual usable aparece en el primer mes completo).

In [189]:
import pandas as pd

# ---------- helpers ----------
def month_start(s):
    """Convierte fechas a inicio de mes (Timestamp)."""
    s = pd.to_datetime(s)
    return s.dt.to_period("M").dt.to_timestamp()   # inicio de mes

def monthly_keep_already_monthly(df, name, date_col="date", value_col="value", how="last"):
    """
    Para series ya mensuales (una obs por mes o pocas):
    - normaliza fecha a inicio de mes
    - si hay duplicados dentro del mes: how = 'last' o 'mean'
    """
    x = df[[date_col, value_col]].copy()
    x[date_col] = pd.to_datetime(x[date_col])
    x["date"] = month_start(x[date_col])
    x = x.sort_values(date_col)

    if how == "mean":
        out = x.groupby("date", as_index=False)[value_col].mean()
    else:
        out = x.groupby("date", as_index=False)[value_col].last()

    out = out.rename(columns={value_col: name})
    return out[["date", name]]

def monthly_from_daily(df, name, date_col="date", value_col="value", how="last"):
    """
    Para series diarias/irregulares:
    - resamplea a MS
    - how='last' (cierre del mes) o how='mean' (promedio mensual)
    """
    x = df[[date_col, value_col]].copy()
    x[date_col] = pd.to_datetime(x[date_col])
    x = x.sort_values(date_col).set_index(date_col)

    if how == "mean":
        out = x.resample("MS")[value_col].mean()
    else:
        out = x.resample("MS")[value_col].last()

    out = out.rename(name).reset_index().rename(columns={date_col: "date"})
    return out[["date", name]]

def monthly_sum_returns(df, name, date_col="date"):
    """
    Retornos log diarios -> retorno log mensual sumando dentro del mes.
    """
    x = df[[date_col, name]].copy()
    x[date_col] = pd.to_datetime(x[date_col])
    x = x.sort_values(date_col).set_index(date_col)
    out = x.resample("MS")[name].sum().reset_index().rename(columns={date_col: "date"})
    return out[["date", name]]

# ---------- 1) retornos mensuales ----------
m_r_spy    = monthly_sum_returns(rets[["date","r_SPY"]], "r_SPY")
m_tech_rel = monthly_sum_returns(rets[["date","tech_rel"]], "tech_rel")

# ---------- 2) macro mensual ----------
# IGAE ya lo tienes como ['date','IGAE']
m_igae = igae.copy()
m_igae["date"] = month_start(m_igae["date"])
m_igae = m_igae[["date","IGAE"]]

# INPC está en series["INPC"] (mensual)
m_inpc = monthly_keep_already_monthly(series["INPC"], "INPC")

# EMBIG (diario en tu CSV de BCRP) -> último del mes
m_embig = monthly_from_daily(series["EMBIG"], "EMBIG", how="last")

m_dgs10 = series["DGS10"], "DGS10"
m_vix   = series["VIXCLS"], "VIXCLS"
m_usdmxn   = series["DEXMXUS"], "DEXMXUS"

# CETES28 (diario/semanal) -> último del mes
#m_cetes  = monthly_from_daily(series["CETES28"], "CETES28", how="last")

# ---------- 3) merge ----------
dfs = [m_igae, m_inpc, m_usdmxn, m_dgs10, m_vix, m_embig, m_r_spy, m_tech_rel]

df_m = dfs[0]
for d in dfs[1:]:
    df_m = df_m.merge(d, on="date", how="outer")

df_m = df_m.sort_values("date").reset_index(drop=True)

# (opcional, recomendado) reindex a todos los meses para que quede una malla perfecta
full_idx = pd.date_range(df_m["date"].min(), df_m["date"].max(), freq="MS")
df_m = (df_m.set_index("date")
            .reindex(full_idx)
            .rename_axis("date")
            .reset_index())

print("Panel mensual:", df_m.shape, "| rango:", df_m["date"].min(), "->", df_m["date"].max())
df_m.head(12), df_m.tail(12)

TypeError: Can only merge Series or DataFrame objects, a <class 'tuple'> was passed

### Integrar CETES28 al panel mensual del VAR

Unimos `cetes_m` al panel mensual `df_m` por `date`.
Después redefinimos la muestra de estimación para que empiece en 2006 (primer mes disponible de CETES) y para evitar NA's.

In [ ]:
dfs = [m_igae, m_inpc, m_usdmxn, m_dgs10, m_vix, m_embig, m_r_spy, m_tech_rel]

df_m = dfs[0]
for d in dfs[1:]:
    df_m = df_m.merge(d, on="date", how="inner")  # 👈 intersección: evita NaNs estructurales

df_m = df_m.sort_values("date").reset_index(drop=True)

# Recorte explícito a muestra objetivo del VAR (2006–2025)
start = pd.Timestamp("2006-01-01")
end   = pd.Timestamp("2025-12-01")  # mensual en "MS" (inicio de mes)
df_m = df_m[(df_m["date"] >= start) & (df_m["date"] <= end)].copy()

print("Panel mensual (INNER, recortado):", df_m.shape, "|", df_m["date"].min().date(), "->", df_m["date"].max().date())
df_m.head(12), df_m.tail(12)

Panel mensual (INNER, recortado): (20, 9) | 2006-01-01 -> 2025-01-01


(         date       IGAE       INPC  value_x  value_y  value  EMBIG     r_SPY  \
 0  2006-01-01  79.264683  60.603626  10.5422     4.42  12.04  132.0  0.027260   
 1  2007-01-01  80.898322  63.016208  10.9559     4.76  11.04  124.0  0.003842   
 2  2008-01-01  82.477618  65.350564  10.9057     3.74  25.82  198.0 -0.098169   
 3  2009-01-01  76.019281  69.456149  13.8839     2.52  44.68  411.0 -0.047104   
 4  2010-01-01  77.817792  72.552046  12.8096     3.73  20.64  214.0 -0.033568   
 5  2011-01-01  81.059540  75.295991  12.1280     3.39  17.32  162.0  0.020543   
 6  2012-01-01  85.357613  78.343049  13.3829     1.97  20.23  229.0  0.046607   
 7  2013-01-01  87.662037  80.892782  12.6964     1.91  13.51  165.0  0.070825   
 8  2014-01-01  88.055324  84.519052  13.2220     2.86  14.24  219.0 -0.038988   
 9  2015-01-01  90.618371  87.110103  14.6972     1.88  19.12  252.0 -0.036366   
 10 2016-01-01  91.735198  89.386381  18.0648     2.09  23.72  362.0 -0.076475   
 11 2017-01-01  

In [ ]:
df_m

,date,IGAE,INPC,value_x,value_y,value,EMBIG,r_SPY,tech_rel
0,2006-01-01,79.264683,60.603626,10.5422,4.42,12.04,132.0,0.027260,0.012228
1,2007-01-01,80.898322,63.016208,10.9559,4.76,11.04,124.0,0.003842,0.004959
2,2008-01-01,82.477618,65.350564,10.9057,3.74,25.82,198.0,-0.098169,-0.063442
3,2009-01-01,76.019281,69.456149,13.8839,2.52,44.68,411.0,-0.047104,0.058754
4,2010-01-01,77.817792,72.552046,12.8096,3.73,20.64,214.0,-0.033568,-0.018477
5,2011-01-01,81.059540,75.295991,12.1280,3.39,17.32,162.0,0.020543,0.001370
6,2012-01-01,85.357613,78.343049,13.3829,1.97,20.23,229.0,0.046607,0.037159
7,2013-01-01,87.662037,80.892782,12.6964,1.91,13.51,165.0,0.070825,-0.020205
8,2014-01-01,88.055324,84.519052,13.2220,2.86,14.24,219.0,-0.038988,0.015928
9,2015-01-01,90.618371,87.110103,14.6972,1.88,19.12,252.0,-0.036366,0.010244


## ✅ Bloque 4 — Transformaciones para estacionariedad (INPC e IGAE)

Objetivo:
- Convertir variables de nivel (INPC e IGAE) a tasas de crecimiento (más estacionarias).
- Para INPC: inflación mensual = 100 * Δlog(INPC).
- Para IGAE: crecimiento mensual de actividad = 100 * Δlog(IGAE).

Estas transformaciones dejan las series en “puntos porcentuales” (aprox.), y son típicas antes de estimar un VAR.

In [ ]:
import numpy as np

# Trabajamos sobre la muestra desde 2005
df_est = df_m[df_m["date"] >= pd.Timestamp("2006-01-01")].copy()

# ---------- transformaciones (Δlog * 100) ----------
# Inflación mensual (aprox % mensual)
df_est["infl_m"] = 100 * (np.log(df_est["INPC"]) - np.log(df_est["INPC"].shift(1)))

# Crecimiento mensual IGAE (aprox % mensual)
df_est["g_igae"] = 100 * (np.log(df_est["IGAE"]) - np.log(df_est["IGAE"].shift(1)))

# (Opcional) FX en cambios % (si quieres, mejor que nivel)
df_est["dlog_fx"] = 100 * (np.log(df_est["DEXMXUS"]) - np.log(df_est["DEXMXUS"].shift(1)))

# Revisa rápidamente
cols_check = [
    "date",
    "IGAE","g_igae",
    "INPC","infl_m",
    "DEXMXUS","dlog_fx",
    "DGS10","VIXCLS","EMBIG",
    "CETES28",   # 👈 agregar
    "tech_rel"
]
df_est[cols_check].head(12), df_est[cols_check].tail(12)

KeyError: 'DEXMXUS'

# VAR

In [ ]:
var_cols = [
    "tech_rel",
    "VIXCLS",
    "DGS10",
    "EMBIG",
    "dlog_fx",
    "CETES28",
    "infl_m",
    "g_igae"
]

df_var = df_est[["date"] + var_cols].dropna().copy()

df_var = df_var.set_index("date")

print("Muestra VAR:", df_var.shape)
df_var.head(), df_var.tail()

Muestra VAR: (0, 8)


(Empty DataFrame
 Columns: [tech_rel, VIXCLS, DGS10, EMBIG, dlog_fx, CETES28, infl_m, g_igae]
 Index: [],
 Empty DataFrame
 Columns: [tech_rel, VIXCLS, DGS10, EMBIG, dlog_fx, CETES28, infl_m, g_igae]
 Index: [])